In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

df = pd.read_csv("diabetes.csv")


X = df.drop(columns=["Class"]).values
y = (df["Class"] == "positive").astype(float).values  # positive -> 1, negative -> 0

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
df.shape

In [ ]:
# Define the model (7 input features in this dataset)
model = nn.Sequential(
    nn.Linear(7, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid(),
)
print(model)

In [ ]:
# THis part to Run only the first layer directly

with torch.no_grad():
    out1 = model[0](X)

print(out1.shape)  # torch.Size([768, 12]) -> 12 values per patient
print(out1[0])  # first patient's 12 raw outputs

In [ ]:
# First layer directly with  ReLU

with torch.no_grad():
    out1_relu = model[:2](X)  # Linear(7,12) -> ReLU
print(out1_relu[0])  # same 12 values but negatives clipped to 0

In [ ]:
# one layer at a time

x = X[0:1]  # first patient (keep 2D shape: [1, 7])

with torch.no_grad():
    for layer in model:
        x = layer(x)
        print(f"{layer.__class__.__name__:>8}: {x.numpy().round(3)}")

In [ ]:
# Forward Pass
with torch.no_grad():
    out_relu = model[:2](X)  # runs model[0] (Linear) then model[1] (ReLU)

print(out_relu.shape)  # torch.Size([768, 12])
print(out_relu[0])  # first patient's 12 activations after ReLU

In [ ]:
# Train the model
loss_fn = nn.BCELoss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 100
batch_size = 10

In [ ]:
for epoch in range(n_epochs):
    for i in range(0, len(X), batch_size):
        Xbatch = X[i : i + batch_size]
        y_pred = model(Xbatch)
        ybatch = y[i : i + batch_size]
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Finished epoch {epoch}, latest loss {loss}")

In [ ]:
def train_with_lr(lr, n_epochs=20, batch_size=10):
    torch.manual_seed(0)
    m = nn.Sequential(
        nn.Linear(7, 12),
        nn.ReLU(),
        nn.Linear(12, 8),
        nn.ReLU(),
        nn.Linear(8, 1),
        nn.Sigmoid(),
    )
    opt = optim.Adam(m.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    history = []
    for epoch in range(n_epochs):
        epoch_loss, n_batches = 0.0, 0
        for i in range(0, len(X), batch_size):
            y_pred = m(X[i : i + batch_size])
            loss = loss_fn(y_pred, y[i : i + batch_size])
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            n_batches += 1
        history.append(epoch_loss / n_batches)

    with torch.no_grad():
        accuracy = (m(X).round() == y).float().mean().item()
    return history, accuracy

In [ ]:
results = {lr: train_with_lr(lr) for lr in [5.0, 1e-7, 0.001]}

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
for lr, (history, _) in results.items():
    plt.plot(range(1, len(history) + 1), history, marker="o", label=f"lr = {lr:g}")
plt.xlabel("Epoch")
plt.ylabel("Mean training loss (BCE)")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("lr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Compute accuracy (no_grad is optional)
with torch.no_grad():
    y_pred = model(X)
accuracy = (y_pred.round() == y).float().mean()
print(f"Accuracy {accuracy}")

In [ ]:
import matplotlib.pyplot as plt


def draw_network(
    layer_sizes,
    layer_labels=None,
    layer_colors=None,
    node_r=0.28,
    x_gap=3.2,
    y_gap=0.85,
    save_as=None,
):
    """Draw a fully-connected neural network as circles and lines."""
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.set_aspect("equal")
    ax.axis("off")

    if layer_colors is None:
        layer_colors = ["#5DCAA5"] + ["#AFA9EC"] * (len(layer_sizes) - 2) + ["#F0997B"]

    # Compute node positions (each layer vertically centered)
    positions = []
    for i, n in enumerate(layer_sizes):
        x = i * x_gap
        ys = [(j - (n - 1) / 2) * y_gap for j in range(n)]
        positions.append([(x, y) for y in ys])

    # Connections first (so circles draw on top)
    for a, b in zip(positions[:-1], positions[1:]):
        for x1, y1 in a:
            for x2, y2 in b:
                ax.plot(
                    [x1, x2],
                    [y1, y2],
                    color="gray",
                    linewidth=0.4,
                    alpha=0.35,
                    zorder=1,
                )

    # Neurons
    for pts, color in zip(positions, layer_colors):
        for x, y in pts:
            ax.add_patch(
                plt.Circle(
                    (x, y),
                    node_r,
                    facecolor=color,
                    edgecolor="#444",
                    linewidth=0.6,
                    zorder=2,
                )
            )

    # Labels above each layer
    if layer_labels:
        top = max(len(n) for n in positions) * 0  # noqa
        y_top = max(max(y for _, y in pts) for pts in positions) + 1.0
        for i, label in enumerate(layer_labels):
            ax.text(
                i * x_gap,
                y_top,
                label,
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold",
            )

    ax.autoscale()
    ax.margins(0.08)
    if save_as:
        fig.savefig(save_as, dpi=200, bbox_inches="tight")
    return fig, ax


if __name__ == "__main__":
    draw_network(
        layer_sizes=[7, 12, 8, 1],
        layer_labels=[
            "Input\n7 features",
            "Hidden 1\nReLU",
            "Hidden 2\nReLU",
            "Output\nSigmoid",
        ],
        save_as="NN1.png",
    )
    print("saved")

In [ ]:
# Q1:
# The weights inside nn.Linear(7, 12) are learnable parameters.
# They start with random values and are updated during training using backpropagation.
# They are adjusted to minimize the loss function, allowing the model to learn patterns in the data.

In [ ]:
# Q2:
# With a too large learning rate (e.g., 5.0), the model's loss may not converge, leading to poor performance.
# With a too small learning rate (e.g., 1e-7), the model may converge very slowly, taking many epochs to reach a good solution.

In [ ]:
# Q3:
# a): 77 updates per epoch
# b): 7700
# c): The last batch only contains 8 samples, which is not a full batch.

In [ ]:
# Q4:
# a): batch=768 is called batch gradient descent (BGD). It uses the entire dataset to compute the gradient and update the model parameters in one step.
# This can lead to more stable convergence but may be slower for large datasets.
# b): batch=1 is called stochastic gradient descent (SGD). It updates the model parameters after each training example, which can lead to faster convergence but may introduce more noise in the updates.

In [ ]:
# Q5:
# The model might be overfitting or underfitting.
# We can use a validation set to find a proper epoch to stop training.
# When the loss on the validation set starts to increase while the training loss continues to decrease, it indicates overfitting.

In [ ]:
# Q6:
# 1. The output range of Sigmoid function is (0, 1), which fits the diabetes binary classification problem.
# The result of the Sigmoid function can be interpreted as the probability of a patient having diabetes.
# 2. The BCE loss function requires the input to be in the range (0, 1).

In [ ]:
# Q7:
# 7 -> 12 -> 8 -> 1
# The number of weights in the first layer is 7*12 + 12 = 96, in the second layer is 12*8 + 8 = 104, and in the third layer is 8*1 + 1 = 9. So the total weights = 96 + 104 + 9 = 209.

In [ ]:
# Q8:

# a):
w = torch.tensor([0.1, -0.2])
b = torch.tensor([0.3])

x = torch.tensor([0.5, 1.0])
z = torch.dot(w, x) + b
print(z)

# b):
print(nn.ReLU(True)(z))

# c):
b = torch.tensor([-0.3])
z = torch.dot(w, x) + b
print(z)
print(nn.ReLU(True)(z))
# At this point, the gradient of ReLU is 0, so the weights will not be updated during backpropagation.
# This neuron is "dead" and will not contribute to learning.

In [ ]:
# Q9:
sigmoid = nn.Sigmoid()

# a):
z = torch.tensor([0])
print(sigmoid(z))

# b):
z = torch.tensor([0.15])
res = sigmoid(z).item()
res = round(res, 3)
print(res)

# c):
# When z -> +INF, sigmoid(z) -> 1, which means the model is very confident that the patient has diabetes.
# When z -> -INF, sigmoid(z) -> 0, which means the model is very confident that the patient does not have diabetes.

In [ ]:
# Q10:
bce_loss = nn.BCELoss()
label = torch.tensor([1.0])
y_hat = torch.tensor([0.9])
loss = bce_loss(y_hat, label)
print("y_hat is 0.9, loss = ", loss.item())

y_hat = torch.tensor([0.1])
loss = bce_loss(y_hat, label)
print("y_hat is 0.1, loss = ", loss.item())

# The result shows the loss is not symmetric.
# When the model predicts a high probability for the positive class (0.9), the loss is low (0.105).
# When it predicts a low probability (0.1), the loss is high (2.302).
# This indicates that the model is penalized more for being wrong when it is confident in its prediction.

In [ ]:
# Q11:
# a): w_new = 0.37 - 1.2*(-0.0201) = 0.37 + 0.02412 = 0.39412
# b): Because the value should be updated in the opposite direction of the gradient.

In [ ]:
# Q12:
# a): ∂E/∂w = ∂E/∂y * ∂y/∂z * ∂z/∂w
# b): ∂z/∂w, taking the partial derivative with respect to w treats x and b as constants, so x remains.
# c): the largest σ' is at z=0, where σ'(z) = σ(z)(1-σ(z)) = 0.25.

In [ ]:
# Q13:
# The weight after batch 3 is g = g1+g2+g3.
# Since the gradient is accumulated over the batch, each gradient contributes to the final update, so the weight is updated based on the average gradient of the batch.

In [ ]:
# Q14:
# Nothing will be updated if `optimizer.step()` were called before `loss.backward()` in the first iteration.
# Since the gradients should be calculated before the optimizer updates the weights.